# 07 Land capacity

Reports the City of Revelstoke's real zoning distribution (by parcel count) and the Resort Lands DPA's area, both queried live from the City's ArcGIS FeatureServers. Does not compute a unit-capacity number: that needs the zoning bylaw's density provisions (units per lot, FAR, minimum lot size per zone), which have not been found as a document this session (see Open issues in steps/07). What this notebook can show is which zones even allow multi-unit housing, and how many parcels carry them.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"

## Zoning distribution (transcribed from claim C049)

Queried live this session via the FeatureServer's statistics endpoint (group by zoningName, count of OBJECTID). Kept as a literal transcription here, same reasoning as notebooks 02/03: one place records what was queried, this notebook works from the recorded result.

In [ ]:
ZONING_COUNTS = """
zoning_code,zoning_name,parcel_count
P-3,Public Utilities Zone,63
MU-1,Downtown Zone,257
R-LD5,Manufactured Home and Ground-Oriented Dwelling Zone,102
R-MD2,Multi-Unit Apartment Zone,6
CD-02,Comprehensive Development Zone 2,93
I-1,Low Impact Zone,63
CD-09,Comprehensive Development Zone 9,1
R-LD4,Manufactured Home Zone,26
R-LD3,Multi-Unit Rowhouse Zone,7
I-2,Medium Impact Zone,12
CD-01,Comprehensive Development Zone 1,2
I-4,Airport Zone,1
MU-2,Downtown Fringe Zone,96
C-2,Urban Tourist Accommodation Zone,22
R-LD6,Tourist Accommodation Zone,94
CD-05,Comprehensive Development Zone 5,2
P-2,Institutional Zone,89
CD-06,Comprehensive Development Zone 6,6
I-5,Rural Recreation and Natural Resource Zone,26
C-1,Highway Commercial Zone,30
R-LD7,Rural Residential Zone,40
C-4,Local Neighbourhood Commercial Zone,7
CD-10,Comprehensive Development Zone 10,27
R-LD2,Small Lot Ground-Oriented Dwelling Zone,85
CD-03,Comprehensive Development Zone 3,9
CD-12,Comprehensive Development Zone 12,4
CD-11,Comprehensive Development Zone 11,11
C-3,Rural Tourist Accommodation Zone,3
CD-04,Comprehensive Development Zone 4,1
R-LD1,Ground-Oriented Dwelling Zone,2891
E-1,Environmental and Rural Recreation Zone,301
R-MD3,Multi-Unit Condominium Zone,47
CD-07,Comprehensive Development Zone 7,1
I-3,High Impact Zone,56
P-1,Parks and Public Use Zone,213
MU-3,Corridor Zone,53
R-MD1,Multi-Unit Rowhouse Zone,10
CD-08,Comprehensive Development Zone 8,5
MU-4,Live-Work Zone,51
"""

In [ ]:
import io
import sys

import pandas as pd

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
assert ledger["claims"]["C049"]["source_id"] == "S045"

zoning = pd.read_csv(io.StringIO(ZONING_COUNTS))
assert zoning["parcel_count"].sum() > 0
zoning.sort_values("parcel_count", ascending=False).head(10)

## Which zones allow multi-unit housing at all

A zone's own name states its use type (rowhouse, apartment, condominium, downtown/corridor/live-work mixed use); this is evidence, read directly off the zone names, not a judgment about density. How many units each such parcel could actually hold is not answered here.

In [ ]:
MULTI_UNIT_ELIGIBLE_PREFIXES = ("R-MD", "R-LD3", "MU-")

In [ ]:
zoning["multi_unit_eligible_by_name"] = zoning["zoning_code"].str.startswith(
    MULTI_UNIT_ELIGIBLE_PREFIXES
)
multi_unit_summary = (
    zoning.groupby("multi_unit_eligible_by_name")["parcel_count"].sum()
)
multi_unit_summary

## Resort Lands DPA area (claim C050)

One polygon, one area figure; kept as plain values rather than a table.

In [ ]:
assert ledger["claims"]["C050"]["source_id"] == "S047"
resort_lands_dpa = {
    "area_sq_m": 8110956.33984375,
    "area_hectares": 8110956.33984375 / 10_000,
}
resort_lands_dpa

## Write outputs

Zoning table and the resort-lands area, so step 08 can cite the same numbers without re-querying the FeatureServers.

In [ ]:
import json
import os

os.makedirs(processed_dir, exist_ok=True)
zoning.to_csv(f"{processed_dir}/07_zoning_by_parcel_count.csv", index=False, encoding="utf-8")
with open(f"{processed_dir}/07_resort_lands_dpa.json", "w", encoding="utf-8") as f:
    json.dump(resort_lands_dpa, f, indent=2)
print("wrote 07_zoning_by_parcel_count.csv, 07_resort_lands_dpa.json")

## Checks

Every parcel must be classified into exactly one of the two multi-unit-eligible groups (no zone left unclassified), and the total parcel count from the grouped sum must match the raw table's own sum, since a mismatch would mean a zone code was silently dropped or double-counted.

In [ ]:
assert multi_unit_summary.sum() == zoning["parcel_count"].sum()
assert zoning["multi_unit_eligible_by_name"].notna().all()
print("checks passed")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))